# Содержание
### Строим граф для 59 сбщ
1. Стереть Neo4j базу до нуля (7688)
2. Вызов DeepSeek с промптом извлечения узлов и связей из сообщения (cклеиваем строки в промпт (+ сбщ, + онто))
3. (Отключено) Валидация и исправление полученного ABox JSON c помощью LLM
4. JSON -> Cypher конвертер (LLM возвращает JSON с узлами и нодами, чтобы в конт окно влезало. Cypher не влезал!) ВАЖНО: он зависит от текущей онтологии. если та поменялась, его тоже меняем
5. Запись в файл cypher_checkpoints - слепки базы на каждое сбщ
6. Внесение инфы в базу (исполнение всех Cypher команд по очереди)
7. Повторить 2-6 пункты для всех сбщ 
   

In [34]:
%pip -q install python-dotenv

from dotenv import load_dotenv
import os

load_dotenv(".env")   # подхватит /.../neo4j_граф/.env

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
print("DEEPSEEK_API_KEY loaded:", bool(DEEPSEEK_API_KEY))


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
DEEPSEEK_API_KEY loaded: True


In [13]:
# 1 стирание базы полное до нуля

In [63]:
def step_1_reset_db():
    !docker stop neo4j_travel || true
    !docker rm neo4j_travel || true
    !docker volume rm neo4j_travel_data || true
    
    !docker run -d --name neo4j_travel \
      -p 7475:7474 -p 7688:7687 \
      -e NEO4J_AUTH=neo4j/12345678 \
      -v neo4j_travel_data:/data \
      neo4j:5
    
    # !docker ps --filter name=neo4j_travel   
    print("Did reset DB")
    return None


neo4j_travel
neo4j_travel
neo4j_travel_data
fc42c2acda3dd862edb1762ad1f2d1e7012b58436ca502402de952e9f2806a21
Did reset DB


In [67]:
# RESET второй базы

def step_1_reset_db_2():
    !docker stop neo4j_travel2 || true
    !docker rm neo4j_travel2 || true
    !docker volume rm neo4j_travel2_data || true

    !docker run -d --name neo4j_travel2 \
      -p 7476:7474 -p 7689:7687 \
      -e NEO4J_AUTH=neo4j/12345678 \
      -v neo4j_travel2_data:/data \
      neo4j:5

    print("Did reset DB #2 (http://localhost:7476, bolt neo4j://localhost:7689)")
    return None

In [66]:
# 2 сборка промпта и запрос к DeepSeek - выход ABox в виде JSON

In [16]:
def step_2_prompt_to_abox(msg_as_JSON) -> str:
    import os
    import json
    import requests
    import time

    PROMPT_PATH = "prompt.txt"
    ONTO_PATH = "ontology.txt"

    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("Set env var DEEPSEEK_API_KEY (export DEEPSEEK_API_KEY=...).")

    DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
    MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")  # or deepseek-chat

    # --- load prompt template + ontology ---
    with open(PROMPT_PATH, "r", encoding="utf-8") as f:
        prompt_template = f.read()

    with open(ONTO_PATH, "r", encoding="utf-8") as f:
        ontology_text = f.read().strip()

    # --- normalize message JSON string ---
    if isinstance(msg_as_JSON, (dict, list)):
        message_json_string = json.dumps(msg_as_JSON, ensure_ascii=False)
    else:
        message_json_string = str(msg_as_JSON).strip()

    # validate that it's JSON (LLM expects JSON string with keys post_id/post_url/author/date/text)
    try:
        json.loads(message_json_string)
    except Exception as e:
        raise ValueError(f"msg_as_JSON must be a valid JSON string (or dict). Parse error: {e}") from e

    # --- stitch prompt ---
    prompt = (
        prompt_template
        .replace("{RDF_ONTOLOGY_HERE}", ontology_text)
        .replace("{MESSAGE_JSON_STRING_HERE}", message_json_string)
    )

    # sanity check for unresolved placeholders
    unresolved = [p for p in ["{RDF_ONTOLOGY_HERE}", "{MESSAGE_JSON_STRING_HERE}"] if p in prompt]
    if unresolved:
        raise RuntimeError(f"Unresolved placeholders in prompt: {unresolved}")

    # --- call DeepSeek ---
    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL,
        "messages": [
            # system можно оставить очень коротким, промпт уже строгий
            {"role": "system", "content": "Return ONLY valid JSON object. No markdown. No code fences."},
            {"role": "user", "content": prompt},
        ],
        "temperature": 0.0,
    }

    t0 = time.perf_counter()
    resp = requests.post(url, headers=headers, json=payload, timeout=180)
    elapsed = time.perf_counter() - t0
    print(f"DeepSeek API latency: {elapsed:.2f}s")
    resp.raise_for_status()
    data = resp.json()

    api_finish = data["choices"][0].get("finish_reason", "")
    content = data["choices"][0]["message"]["content"]

    emoji = "🟢" if api_finish != "length" else "🔴"
    print(f"{emoji} API finish_reason = {api_finish}")

    # optional: validate JSON (hard fail if model returned junk)
    try:
        json.loads(content)
    except Exception as e:
        raise ValueError(f"Model did not return valid JSON. Error: {e}\nRaw content:\n{content[:2000]}") from e

    print(content)  # preview
    return content


In [17]:
# print(answer)  # preview
# print(prompt)

In [18]:
# 3 Валидация ABox JSON перед генерацией Cypher

In [19]:
def step_3_validate_abox():
    
    # import json
    # import re
    
    # def _strip_code_fences(s: str) -> str:
    #     s = s.strip()
    #     if s.startswith("```"):
    #         s = re.sub(r"^```(?:json)?\s*", "", s)
    #         s = re.sub(r"\s*```$", "", s)
    #     return s.strip()
    
    # def _one_line(s: str) -> str:
    #     return " ".join(str(s).split())
    
    # raw_answer = _strip_code_fences(answer)
    # try:
    #     abox_obj = json.loads(raw_answer)
    #     abox_json = json.dumps(abox_obj, ensure_ascii=False, separators=(",", ":"))
    # except Exception:
    #     abox_json = _one_line(raw_answer)
    
    # message = message_text
    # ontology = ontology_text
    
    # PROMPT_VALID_PATH = "prompt_extr_valid.txt"
    # with open(PROMPT_VALID_PATH, "r", encoding="utf-8") as f:
    #     prompt_template_valid = f.read()
    
    # prompt_validation = (prompt_template_valid
    #     .replace("{MESSAGE}", _one_line(message))
    #     .replace("{ONTOLOGY}", _one_line(ontology))
    #     .replace("{ABOX_JSON}", abox_json)
    # )
    
    
    # payload_valid = {
    #     "model": MODEL,
    #     "messages": [
    #         {"role": "system", "content": "Return ONLY JSON. No markdown. Output must be a valid ABox JSON and nothing else."},
    #         {"role": "user", "content": prompt_validation},
    #     ],
    #     "temperature": 0.0,
    # }
    
    # resp_valid = requests.post(url, headers=headers, json=payload_valid, timeout=180)
    # resp_valid.raise_for_status()
    # data_valid = resp_valid.json()
    # api_finish_valid = data_valid["choices"][0].get("finish_reason", "")
    # content_valid = data_valid["choices"][0]["message"]["content"]
    
    # answer_raw = answer
    # answer = content_valid
    
    # emoji = "🟢" if api_finish_valid != "length" else "🔴"
    # print(f"{emoji} API finish_reason = {api_finish_valid}")
    # print(answer)
    return None


In [20]:
# 4 JSON → Cypher генерация

In [24]:
def step_4_abox_to_cypher(LLM_answer: str) -> List[str]:
    import json
    import re

    # =========================
    # Helpers
    # =========================

    def strip_code_fences(s: str) -> str:
        s = (s or "").strip()
        if s.startswith("```"):
            s = re.sub(r"^```(?:json)?\s*", "", s)
            s = re.sub(r"\s*```$", "", s)
        return s.strip()

    def neo4j_local_name(name: str) -> str:
        if not name:
            return ""
        if ":" in name:
            return name.split(":")[-1]
        if "#" in name:
            return name.split("#")[-1]
        if "/" in name:
            return name.rsplit("/", 1)[-1]
        return name

    def neo4j_safe_ident(name: str) -> str:
        n = neo4j_local_name(name)
        n = re.sub(r"[^A-Za-z0-9_]", "_", n)
        if not n:
            return "X"
        if n[0].isdigit():
            n = "_" + n
        return n

    def cypher_literal(v, prop_name=None):
        if v is None:
            return "null"
        if isinstance(v, bool):
            return "true" if v else "false"
        if isinstance(v, (int, float)):
            return str(v)

        # New ontology/prompt: datetime is stored as ISO8601 string in pcfr:sourcePostDateTime
        if prop_name in {"sourcePostDateTime"}:
            # Expecting something like "2013-02-06T06:19:00"
            return f"localdatetime('{str(v)}')"

        s = str(v)
        s = s.replace("\\", "\\\\").replace("'", "\\'")
        s = s.replace("\n", "\\n").replace("\r", "\\r")
        return f"'{s}'"

    def contains_cyrillic(s: str) -> bool:
        return bool(re.search(r"[А-Яа-яЁё]", s or ""))

    # =========================
    # Parse JSON from LLM
    # =========================

    raw = strip_code_fences(LLM_answer)
    root = json.loads(raw)

    payload = root.get("payload", {}) or {}
    nodes = payload.get("nodes", []) or []
    rels = payload.get("relationships", []) or []

    cypher_statements = []

    # =========================
    # Constraints
    # =========================

    labels = sorted({
        neo4j_safe_ident(n.get("label", ""))
        for n in nodes
        if n.get("label")
    })

    for label in labels:
        cypher_statements.append(
            f"CREATE CONSTRAINT IF NOT EXISTS FOR (n:{label}) REQUIRE n.key IS UNIQUE"
        )

    # =========================
    # Nodes
    # =========================

    for node in nodes:
        label_raw = node.get("label", "")
        key = node.get("key")

        if not label_raw or key is None:
            continue

        label = neo4j_safe_ident(label_raw)
        props = node.get("properties", {}) or {}

        set_parts = []

        for k, v in props.items():
            prop_key = neo4j_safe_ident(k)

            # New prompt: output should be Latin-only, but keep a warning just in case.
            if isinstance(v, str) and contains_cyrillic(v):
                print(f"⚠ WARNING: Cyrillic detected in property '{prop_key}' → '{v}'")

            set_parts.append(f"n.{prop_key}={cypher_literal(v, prop_key)}")

        set_clause = (" SET " + ", ".join(set_parts)) if set_parts else ""
        stmt = f"MERGE (n:{label} {{key:{cypher_literal(key)}}}){set_clause}"
        cypher_statements.append(stmt)

    # =========================
    # Relationships
    # =========================

    for rel in rels:
        f = rel.get("from", {}) or {}
        t = rel.get("to", {}) or {}

        fl_raw = f.get("label", "")
        fk = f.get("key")
        tl_raw = t.get("label", "")
        tk = t.get("key")
        rt_raw = rel.get("type", "")

        if not fl_raw or not tl_raw or not rt_raw or fk is None or tk is None:
            continue

        fl = neo4j_safe_ident(fl_raw)
        tl = neo4j_safe_ident(tl_raw)
        rt = neo4j_safe_ident(rt_raw)

        stmt = (
            f"MATCH (a:{fl} {{key:{cypher_literal(fk)}}}), "
            f"(b:{tl} {{key:{cypher_literal(tk)}}}) "
            f"MERGE (a)-[:{rt}]->(b)"
        )
        cypher_statements.append(stmt)

    # =========================
    # Output
    # =========================

    print("LLM finish_reason:", root.get("finish_reason"))
    print("LLM reason:", root.get("reason"))
    print(f"Prepared {len(cypher_statements)} Cypher statements.\n")
    return cypher_statements

In [25]:
# 5 Сохранение Cypher в чекпоинт

In [26]:
def step_5_save_cypher(cypher_statements: List[str]):
    import os
    import re
    
    ckpt_dir = "cypher_checkpoints"
    os.makedirs(ckpt_dir, exist_ok=True)
    
    existing = []
    for name in os.listdir(ckpt_dir):
        m = re.match(r"^cypher_(\d+)\.txt$", name)
        if m:
            existing.append(int(m.group(1)))
    
    next_num = max(existing, default=0) + 1
    out_path = os.path.join(ckpt_dir, f"cypher_{next_num}.txt")
    
    with open(out_path, "w", encoding="utf-8") as f:
        for stmt in cypher_statements:
            f.write(stmt)
            f.write(";")
    
    print(f"Saved {len(cypher_statements)} statements to {out_path}")
    return None


In [27]:
# print(cypher_statements)

In [28]:
# 6 исполнение Cypher (каждая строка автономна)

In [29]:
def step_6_execute_cypher(cypher_statements: List[str]):
    from neo4j import GraphDatabase
    
    URI = "neo4j://localhost:7688"
    USER = "neo4j"
    PASSWORD = "12345678"
    DATABASE = "neo4j"
    
    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
    
    try:
        with driver.session(database=DATABASE) as session:
            for i, stmt in enumerate(cypher_statements, 1):
                session.run(stmt).consume()
        print(f"✅ Executed {len(cypher_statements)} statements.")
    finally:
        driver.close()    
    return None


In [ ]:
# 7 запуск пайплайна построения графа

In [31]:
step_1_reset_db()

import json

with open("messages.json", "r", encoding="utf-8") as f:
    messages = json.load(f)
messages = list(map(lambda x: json.dumps(x, ensure_ascii=False), messages))


for msg in messages:
    print(msg)
    answer = step_2_prompt_to_abox(msg) 
    # step_3_validate_abox()
    cypher_statements = step_4_abox_to_cypher(answer)
    step_5_save_cypher(cypher_statements)
    step_6_execute_cypher(cypher_statements)

# subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


neo4j_travel
neo4j_travel
neo4j_travel_data
953258c578b422700a1544dbfefc5cfc37b7e87da2c9902189cb6be41e52047d
Did reset DB
{"post_id": 3902451, "post_url": "https://forum.awd.ru/viewtopic.php?p=3902451#p3902451", "author": "!Casper!", "date": "11 июн 2013, 09:05", "text": "Прилетали вдвоем с другом в Париж, ШДГ, друг прошел контроль за 10 сек максиму без каких либо вопросов, а вот мне устроили допрос минут на 5. Попросили показать обратный билет, показать страховку, бронь отеля, спросили оплачен ли отель, показать кредитные карты, наличные деньги, спросили сколько наличности при себе, спросили какой картой я оплачивал отель, цель визита, на сколько дней приехал, кем я работаю в России. После этого поставили штамп в паспорт! Вывод: держите все распечатки билетов, броней отеля, страховки и тд при себе, так как могут возникнуть проблемы!"}
DeepSeek API latency: 323.83s
🟢 API finish_reason = stop
{
  "payload": {
    "ontology_version": "pcfr-v4",
    "nodes": [
      {
        "label": "Ci

In [ ]:
было 93 104 97 113 для ШДГ и 25 23 для малого 

In [ ]:
# 8 RAG: LLM -> Cypher -> facts -> LLM


In [8]:
def rag_answer(user_query: str) -> None:
    from neo4j import GraphDatabase
    import os
    import requests
    import json

    # -----------------------------
    # Config
    # -----------------------------
    URI = "neo4j://localhost:7688"
    USER = "neo4j"
    PASSWORD = "12345678"
    DATABASE = "neo4j"

    DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
    if not DEEPSEEK_API_KEY:
        raise RuntimeError("Set env var DEEPSEEK_API_KEY first.")

    DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
    MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")

    url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
    headers = {
        "Authorization": f"Bearer {DEEPSEEK_API_KEY}",
        "Content-Type": "application/json",
    }

    # -----------------------------
    # Helper: call DeepSeek
    # -----------------------------
    def call_llm(system_prompt: str, user_prompt: str, temperature: float = 0.0) -> str:
        payload = {
            "model": MODEL,
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            "temperature": temperature,
        }

        resp = requests.post(url, headers=headers, json=payload, timeout=180)
        resp.raise_for_status()
        data = resp.json()
        return data["choices"][0]["message"]["content"]

    # =========================================================
    # 0️⃣ Extract DB schema (real schema injection)
    # =========================================================

    driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

    try:
        with driver.session(database=DATABASE) as session:

            labels = session.run(
                "CALL db.labels() YIELD label RETURN collect(label) AS labels"
            ).single()["labels"]

            rel_types = session.run(
                "CALL db.relationshipTypes() YIELD relationshipType "
                "RETURN collect(relationshipType) AS relTypes"
            ).single()["relTypes"]

            label_props = {}
            for label in labels:
                result = session.run(
                    f"MATCH (n:{label}) RETURN keys(n) AS keys LIMIT 1"
                ).single()
                label_props[label] = result["keys"] if result else []

    finally:
        driver.close()

    schema_text = (
        f"Available node labels: {labels}\n"
        f"Available relationship types: {rel_types}\n"
        f"Node properties by label: {json.dumps(label_props)}\n"
    )

    # =========================================================
    # 1️⃣ Generate Cypher (STRICT schema-aware prompt)
    # =========================================================

    system_cypher = """
You are a Neo4j Cypher generator.

INPUT YOU WILL RECEIVE:
- Database schema text (labels, relationship types, property keys).
- A user question.

HARD RULES (MUST FOLLOW):
1) Use ONLY the labels, relationship types, and property keys present in the provided schema text.
2) Do NOT invent any labels, relationship types, or properties.
3) Read-only queries only: MATCH / OPTIONAL MATCH / WHERE / WITH / RETURN.
   - Do NOT use CREATE, MERGE, DELETE, SET, CALL.
4) Always include LIMIT.
   - Use LIMIT 50 for list outputs.
   - Use LIMIT 1 for aggregated outputs.
5) If filtering by real-world entities (France, Paris, CDG),
   use entityName or key only if visible in schema sample.
   Never guess key format.
6) Be robust:
   - Use OPTIONAL MATCH when appropriate.
   - Use DISTINCT to avoid double counting.
7) If questioned about country, check all the cities of the country to give nodes and links
FREQUENCY RULE (CRITICAL):
If the question asks about frequency / how often / percentage:
- Compute:
    total_count   = total relevant Encounter in scope
    match_count   = number satisfying condition
    share         = toFloat(match_count) / total_count
- Return ALL THREE fields.

OUTPUT FORMAT:
Return ONLY valid JSON:
{"cypher": "<ONE Cypher query>"}
No markdown. No explanation.
"""

    prompt_cypher = (
        f"Database schema:\n{schema_text}\n\n"
        f"User question:\n{user_query}\n\n"
        "Generate Cypher query."
    )

    cypher_json = call_llm(system_cypher, prompt_cypher, temperature=0.0)

    try:
        cypher_obj = json.loads(cypher_json)
        cypher = cypher_obj.get("cypher", "").strip()
    except Exception:
        cypher = ""

    # =========================================================
    # 2️⃣ Execute Cypher
    # =========================================================

    facts = []

    if cypher:
        driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
        try:
            with driver.session(database=DATABASE) as session:
                records = session.run(cypher).data()
                facts.extend(records)
        finally:
            driver.close()

    facts_text = json.dumps(facts, ensure_ascii=False)

    # =========================================================
    # 3️⃣ Generate Final Answer
    # =========================================================

    system_answer = (
        "You are a concise assistant.\n"
        "If the query result contains total_count, match_count and share, "
        "interpret them as frequency statistics.\n"
        "Use ONLY the provided database facts.\n"
        "If no relevant data exists, say you do not know."
    )

    prompt_answer = (
        f"Question:\n{user_query}\n\n"
        f"Facts:\n{facts_text}\n\n"
        "Answer:"
    )

    final_answer = call_llm(system_answer, prompt_answer, temperature=0.2)

    print("\n--- Injected Schema ---")
    print(schema_text)

    print("\n--- Generated Cypher ---")
    print(cypher)

    print("\n--- Retrieved Facts ---")
    print(facts_text)

    print("\n--- Final Answer ---")
    print(final_answer)

In [8]:
# 9 Проверка RAG

In [10]:
import subprocess
# rag_answer('Является ли паспортный контроль в Париже строгим?')
# rag_answer('Is passport control in Paris strict?')
# rag_answer('Is passport control in Paris takes much time?')
# rag_answer('How often is return ticket required on passport control in France?')
rag_answer('Give me IDs of messages about encounters where return ticket was required on passport control in France')

import subprocess
subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)



--- Injected Schema ---
Available node labels: ['Advice', 'Airport', 'Cash', 'CashAmount', 'City', 'Claim', 'Country', 'DocumentCheck', 'DocumentScrutiny', 'Employment', 'Encounter', 'Experienced', 'FirstPersonExperience', 'ForeignPassport', 'HotelBookingDoc', 'HotelPaymentCard', 'HotelPaymentStatus', 'Outcome', 'PaymentCard', 'Questioning', 'ReturnTicket', 'SecondaryInterrogation', 'Stamping', 'TravelInsurance', 'Traveller', 'UniversalScope', 'VisitDuration', 'VisitPurpose', 'GeneralKnowledge', 'Generic', 'HabitualScope', 'ProcessingSpeed', 'StampingPractice', 'VerbalInquiryOnly', 'BorderOfficer', 'Printout', 'PhysicalDocumentRequested', 'Required', 'Heard', 'SingleEventScope', 'NoExtraProcedures', 'PassportWithVisa', 'Inferred']
Available relationship types: ['locatedInCity', 'locatedInCountry', 'atAirport', 'atCity', 'atCountry', 'hasTraveler', 'hasScreeningStatus', 'assertionKind', 'evidentiality', 'hasQuestioning', 'hasStamping', 'hasOutcome', 'hasDocumentCheck', 'requestedDocTyp

CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

# Секция: Пайплайн вычисления метрик по датасету (вопрос; триплеты)

ВАЖНО: в базе ровно 1 сбщ ШДГ.

1. на диске: gold_dataset_1.json - файл с моими проверенными глазами триплетами по 1 сообщению ШДГ
2. LLM -> Cypher (особая выдача указана в промпте) -> JSON c ключами s p o -> приведение триплетов к виду строк
3. Приведение золотых триплетов к виду строк и вычисление Метрик Hits@k

In [75]:
# 10  вопрос → (schema inject) → DeepSeek → Cypher (JSON)

In [76]:
# === Cell 1: Ask DeepSeek to generate Cypher that returns triples ===

user_query = "Which documents do I need to bring to passport control in Paris?"

import os, json, requests
from pathlib import Path
from neo4j import GraphDatabase

# ---- Neo4j config ----
URI = "neo4j://localhost:7688"
USER = "neo4j"
PASSWORD = "12345678"
DATABASE = "neo4j"

# ---- DeepSeek config ----
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise RuntimeError("DEEPSEEK_API_KEY is not set (load .env first).")

DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL", "https://api.deepseek.com").rstrip("/")
MODEL = os.getenv("DEEPSEEK_MODEL", "deepseek-reasoner")

# ---- Load prompt template ----
prompt_tpl = Path("prompt_retrieval.txt").read_text(encoding="utf-8")

# ---- Extract DB schema text ----
driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
try:
    with driver.session(database=DATABASE) as session:
        labels = session.run("CALL db.labels() YIELD label RETURN collect(label) AS labels").single()["labels"]
        rel_types = session.run(
            "CALL db.relationshipTypes() YIELD relationshipType RETURN collect(relationshipType) AS relTypes"
        ).single()["relTypes"]

        label_props = {}
        for label in labels:
            r = session.run(f"MATCH (n:{label}) RETURN keys(n) AS keys LIMIT 1").single()
            label_props[label] = r["keys"] if r else []
finally:
    driver.close()

schema_text = (
    f"Available node labels: {labels}\n"
    f"Available relationship types: {rel_types}\n"
    f"Node properties by label: {json.dumps(label_props, ensure_ascii=False)}\n"
)

# ---- Fill prompt in memory (do not modify file) ----
prompt = (
    prompt_tpl
    .replace("{SCHEMA_TEXT}", schema_text)
    .replace("{USER_QUERY}", user_query)
)

# ---- Call DeepSeek ----
url = f"{DEEPSEEK_BASE_URL}/v1/chat/completions"
headers = {"Authorization": f"Bearer {DEEPSEEK_API_KEY}", "Content-Type": "application/json"}
payload = {
    "model": MODEL,
    "messages": [{"role": "user", "content": prompt}],
    "temperature": 0.0,
}

resp = requests.post(url, headers=headers, json=payload, timeout=180)
resp.raise_for_status()
data = resp.json()

finish_reason = data["choices"][0].get("finish_reason", "")
content = data["choices"][0]["message"]["content"]

print("finish_reason:", finish_reason)
print("raw LLM output:\n", content)

cypher = json.loads(content)["cypher"].strip()
print("\n--- Generated Cypher ---\n", cypher)

finish_reason: stop
raw LLM output:
 {"cypher":"MATCH (encounter:Encounter)-[r:hasDocumentCheck]->(docCheck:DocumentCheck)\nWHERE (encounter)-[:atCity]->(:City {entityName: 'Paris'})\n  OR (encounter)-[:atAirport]->(:Airport)-[:locatedInCity]->(:City {entityName: 'Paris'})\nWITH DISTINCT encounter, docCheck\nMATCH (docCheck)-[r2:requestedDocType]->(docType)\nRETURN DISTINCT\n  labels(docCheck)[0] AS s_label,\n  docCheck.key AS s_key,\n  type(r2) AS p,\n  labels(docType)[0] AS o_label,\n  docType.key AS o_key\nLIMIT 200"}

--- Generated Cypher ---
 MATCH (encounter:Encounter)-[r:hasDocumentCheck]->(docCheck:DocumentCheck)
WHERE (encounter)-[:atCity]->(:City {entityName: 'Paris'})
  OR (encounter)-[:atAirport]->(:Airport)-[:locatedInCity]->(:City {entityName: 'Paris'})
WITH DISTINCT encounter, docCheck
MATCH (docCheck)-[r2:requestedDocType]->(docType)
RETURN DISTINCT
  labels(docCheck)[0] AS s_label,
  docCheck.key AS s_key,
  type(r2) AS p,
  labels(docType)[0] AS o_label,
  docType.key

In [77]:
# 11 выполнить Cypher → получить triples.json в формате {s,p,o}

In [78]:
# === Cell 2: Execute Cypher and convert result rows -> triples JSON (s,p,o) ===

import json
from neo4j import GraphDatabase

def records_to_spo_triples(records):
    """
    Expect each record to have:
      s_label, s_key, p, o_label, o_key
    Return list of triples in gold-like JSON format.
    """
    triples = []
    for r in records:
        triples.append({
            "s": {"label": r["s_label"], "key": r["s_key"]},
            "p": r["p"],
            "o": {"label": r["o_label"], "key": r["o_key"]},
        })
    return triples

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))
try:
    with driver.session(database=DATABASE) as session:
        records = session.run(cypher).data()
finally:
    driver.close()

retrieved_triples = records_to_spo_triples(records)

print(f"Retrieved rows: {len(records)}")
print("First 5 triples:")
print(json.dumps(retrieved_triples, ensure_ascii=False, indent=2))



Retrieved rows: 5
First 5 triples:
[
  {
    "s": {
      "label": "DocumentCheck",
      "key": "doccheck_encounter_3902451_author_1_returnticket"
    },
    "p": "requestedDocType",
    "o": {
      "label": "ReturnTicket",
      "key": "returnticket"
    }
  },
  {
    "s": {
      "label": "DocumentCheck",
      "key": "doccheck_encounter_3902451_author_1_insurance"
    },
    "p": "requestedDocType",
    "o": {
      "label": "TravelInsurance",
      "key": "travelinsurance"
    }
  },
  {
    "s": {
      "label": "DocumentCheck",
      "key": "doccheck_encounter_3902451_author_1_hotel"
    },
    "p": "requestedDocType",
    "o": {
      "label": "HotelBookingDoc",
      "key": "hotelbookingdoc"
    }
  },
  {
    "s": {
      "label": "DocumentCheck",
      "key": "doccheck_encounter_3902451_author_1_creditcard"
    },
    "p": "requestedDocType",
    "o": {
      "label": "PaymentCard",
      "key": "paymentcard"
    }
  },
  {
    "s": {
      "label": "DocumentCheck",
      

In [ ]:
# 12 gold JSON → canonical strings, retrieved → canonical strings, Hits@k

In [80]:
import json
from pathlib import Path
import subprocess

def triple_to_canon_str(t):
    # case 1: already a string
    if isinstance(t, str):
        return t.strip()

    # case 2: dict in {s,p,o}
    if isinstance(t, dict) and "s" in t and "p" in t and "o" in t:
        return f"{t['s']['label']}|{t['s']['key']}::{t['p']}::{t['o']['label']}|{t['o']['key']}"

    raise TypeError(f"Unsupported triple format: {type(t)} | sample={repr(t)[:200]}")

gold_path = Path("gold_dataset_1.json")
gold_data = json.loads(gold_path.read_text(encoding="utf-8"))

# gold_data может быть:
# - list of triples
# - {"triples":[...]}  (если у тебя обёртка)
gold_triples = gold_data["triples"] if isinstance(gold_data, dict) and "triples" in gold_data else gold_data

gold_set = {triple_to_canon_str(t) for t in gold_triples}
retrieved_set = {triple_to_canon_str(t) for t in retrieved_triples}  # retrieved_triples из ячейки 2

tp = len(gold_set & retrieved_set)
fp = len(retrieved_set - gold_set)
fn = len(gold_set - retrieved_set)

precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / (tp + fn) if (tp + fn) else 0.0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0

print("Gold size:", len(gold_set))
print("Retrieved size:", len(retrieved_set))
print("TP:", tp, "FP:", fp, "FN:", fn)
print(f"Precision = {precision:.3f}")
print(f"Recall     = {recall:.3f}")
print(f"F1         = {f1:.3f}")

subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)

Gold size: 13
Retrieved size: 5
TP: 5 FP: 0 FN: 8
Precision = 1.000
Recall     = 0.385
F1         = 0.556


CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

In [42]:
print(gold_set)
print(retrieved_list)


{'DocumentCheck|doccheck_encounter_3902451_author_1_hotel::requestedDocType::HotelBookingDoc|hotelbookingdoc', 'Encounter|encounter_3902451_author_1::hasDocumentCheck::DocumentCheck|doccheck_encounter_3902451_author_1_creditcard', 'DocumentCheck|doccheck_encounter_3902451_author_1_creditcard::requestedDocType::PaymentCard|paymentcard', 'Encounter|encounter_3902451_author_1::atCity::City|city_paris', 'DocumentCheck|doccheck_encounter_3902451_author_1_returnticket::requestedDocType::ReturnTicket|returnticket', 'Encounter|encounter_3902451_author_1::hasDocumentCheck::DocumentCheck|doccheck_encounter_3902451_author_1_returnticket', 'DocumentCheck|doccheck_encounter_3902451_author_1_insurance::requestedDocType::TravelInsurance|travelinsurance', 'Airport|airport_cdg::locatedInCity::City|city_paris', 'DocumentCheck|doccheck_encounter_3902451_author_1_cash::requestedDocType::Cash|cash', 'Encounter|encounter_3902451_author_1::hasDocumentCheck::DocumentCheck|doccheck_encounter_3902451_author_1_i

In [ ]:
# ??? Запись в базу уже из файла с диска 

In [1]:
from __future__ import annotations
from pathlib import Path
from typing import Optional, Iterable
from neo4j import GraphDatabase


def _split_cypher_statements(text: str) -> list[str]:
    """
    Split on semicolons, ignoring empty chunks.
    (Assumes your generator doesn't put ';' inside string literals.)
    """
    parts = [p.strip() for p in text.split(";")]
    return [p for p in parts if p]


def run_cypher_file(
    cypher_path: str,
    uri: str = "neo4j://localhost:7688",
    user: str = "neo4j",
    password: str = "12345678",
    database: Optional[str] = None,
) -> None:
    """
    Reads Cypher statements from a file and executes them sequentially.
    Each statement is run as its own query (Neo4j requires 1 statement per run()).
    """
    path = Path(cypher_path)
    if not path.exists():
        raise FileNotFoundError(f"Cypher file not found: {path}")

    cypher_text = path.read_text(encoding="utf-8")
    statements = _split_cypher_statements(cypher_text)

    if not statements:
        print("No Cypher statements found in file.")
        return

    driver = GraphDatabase.driver(uri, auth=(user, password))
    try:
        with driver.session(database=database) as session:
            for i, stmt in enumerate(statements, start=1):
                session.run(stmt).consume()  # consume forces execution now
                # print(f"✅ {i}/{len(statements)} executed")
        print(f"✅ Executed {len(statements)} statements from {path.name}")
    finally:
        driver.close()

# run_cypher_file("cypher_checkpoints/cypher_1.txt", uri="neo4j://localhost:7688", user="neo4j", password="12345678")

# Очистить базу 2 (secondary) и внести 1 cypher

In [167]:
step_1_reset_db_2()

import time
time.sleep(9)

first = 1
last = 3
for i in range(first, last + 1):  # от 1 до посл-1 включительно
    run_cypher_file(
        f"cypher_checkpoints/cypher_{i}.txt",
        uri="neo4j://localhost:7689",
        user="neo4j",
        password="12345678"
    )

# a = 3
# for i in range(a, a + 1):  # от 1 до 11 включительно
#     run_cypher_file(
#         f"cypher_checkpoints/cypher_{i}.txt",
#         uri="neo4j://localhost:7689",
#         user="neo4j",
#         password="12345678"
#     )
import subprocess
subprocess.run(["afplay", "/System/Library/Sounds/Blow.aiff"], check=False)


neo4j_travel2
neo4j_travel2
neo4j_travel2_data
7d0dfb6b154eed6f93b0194b1588ba0b3afaac5119f27d410d4ff611acd8a600
Did reset DB #2 (http://localhost:7476, bolt neo4j://localhost:7689)
✅ Executed 69 statements from cypher_1.txt
✅ Executed 34 statements from cypher_2.txt
✅ Executed 43 statements from cypher_3.txt


CompletedProcess(args=['afplay', '/System/Library/Sounds/Blow.aiff'], returncode=0)

In [ ]:
# изменени е имен файлов на -3 каждый

In [107]:
from pathlib import Path
import re

src = Path("cypher_checkpoints")
dst = src.parent / "cypher_checkpoints_shifted"
dst.mkdir(exist_ok=True)

for f in sorted(src.glob("cypher_*.txt")):
    m = re.fullmatch(r"cypher_(\d+)\.txt", f.name)
    if not m:
        continue
    old_n = int(m.group(1))
    new_n = old_n - 3
    if new_n < 0:
        raise ValueError(f"Negative index for {f.name}: {new_n}")

    out = dst / f"cypher_{new_n}.txt"
    if out.exists():
        raise FileExistsError(f"Target already exists: {out}")
    f.replace(out)  # move (use shutil.copy2 if you want copy instead)